# 04. Team Persistence

Notebooks 01–03 introduced the moving parts of an Arc team: formation (`TeamConfig`, `EntityRegistry`), task distribution (`Channel`, `Priority`, `MsgType`), and inter-agent messaging (`MessagingService`, `Cursor`). Everything in those notebooks ran cleanly in process.

This notebook is about what happens when the process stops.

When an agent crashes mid-task, when an audit chain has to survive an OS reboot, when tomorrow's investigator needs to read today's messages — the team has to remember. That memory has to be tamper-evident, recoverable, and isolated per-agent. ArcTeam ships a **storage abstraction** (`StorageBackend` Protocol with `FileBackend` and `MemoryBackend`), a **shared file store** (`TeamFileStore`), a **knowledge graph facade** (`TeamMemoryService` over `TeamMemoryConfig`), and a **chained audit log** (`AuditLogger` with `verify_chain`). All four collaborate on one job: *the team survives a restart with its history intact*.

You will learn:

- Why persistence is a federal-tier requirement, not a nice-to-have
- The `StorageBackend` Protocol and how `MemoryBackend` and `FileBackend` interchange
- How `FileBackend` achieves atomicity (temp + rename) and concurrency-safe JSONL appends (`fcntl.flock`)
- How `TeamFileStore` isolates per-agent shared workspaces and rejects path traversal
- How `TeamMemoryService` + `TeamMemoryConfig` manage the team knowledge graph as a Null Object when disabled
- How `AuditLogger` chains HMACs across records and verifies the chain after restart
- The recovery flow: cold backend → load last seq → append → verify
- Operational guidance: backups, atomic writes, file permissions, HMAC key rotation

Cross-references:

- `01-team-formation.ipynb` introduced `TeamConfig` and the `EntityRegistry`. We assume that surface.
- `02-task-distribution.ipynb` covered `Channel`, `MsgType`, `Priority`. We won't repeat the routing demo.
- `03-messaging-channels.ipynb` walked `MessagingService` end-to-end. We use audit alongside it but won't re-derive the polling protocol.

This notebook is **mock-first**: every cell runs without a network or a real `~/.arc` directory. We use `tmp_path`-style temp directories and `MemoryBackend` throughout.

## 1. Setup

Standard Arc walkthrough preamble: locate the repo root and add every package's `src/` to `sys.path`. No API keys are needed — every example below is mock-first.

In [ ]:
# Setup: make Arc packages importable from this notebook
import os
import sys
from pathlib import Path

from dotenv import load_dotenv

_here = Path.cwd()
for _p in [_here, *_here.parents]:
    if (_p / "packages").is_dir() and (_p / "pyproject.toml").is_file():
        REPO_ROOT = _p
        break
else:
    raise RuntimeError("Could not locate Arc repo root")

# Add every package's src/ (and tests/, where present) to sys.path
for _pkg in (REPO_ROOT / "packages").iterdir():
    for _sub in ("src", "tests"):
        _path = _pkg / _sub
        if _path.is_dir() and str(_path) not in sys.path:
            sys.path.insert(0, str(_path))

load_dotenv(REPO_ROOT / ".env")

Smoke-import the public persistence surface. If any of these fail, the rest of the notebook will not run — fix `sys.path` first.

In [ ]:
import arcteam
from arcteam import (
    AuditLogger,
    AuditRecord,
    FileBackend,
    MemoryBackend,
    StorageBackend,
    TeamConfig,
    TeamFileStore,
    TeamMemoryConfig,
    TeamMemoryService,
)
from arcteam.memory.types import (
    Classification,
    EntityMetadata,
)

print("arcteam version:", arcteam.__version__)
print("Public persistence surface imported OK.")

We will use a tiny helper that returns a fresh temp directory we own. In real test code, pytest's `tmp_path` fixture does this for you; in a notebook we have to allocate one explicitly so each cell that needs disk-backed storage starts clean.

In [ ]:
import tempfile


def fresh_tmp(label: str = "") -> Path:
    """Return a brand-new temp directory. The OS reaps it on shutdown."""
    return Path(tempfile.mkdtemp(prefix=f"arcteam-walkthrough-{label}-"))


DEMO_ROOT = fresh_tmp("demo")
print("Demo root:", DEMO_ROOT)
print("Exists:", DEMO_ROOT.exists(), "  Empty:", not any(DEMO_ROOT.iterdir()))

## 2. Why persistence — survival across restarts, audit, recovery

Three real-world failure modes drive ArcTeam's persistence design:

1. **Process crash mid-poll.** An agent receives a message, starts working, then segfaults before acknowledging. On restart it must see *the same message again* — the cursor must persist on disk, not in memory.
2. **Audit reconstruction after the fact.** A compliance officer six months from now needs to answer "who told the procurement agent to dispatch task X, and at what time?" The log must be append-only, ordered, and tamper-evident.
3. **Shared workspace handoff.** Agent A produces a CSV; agent B picks it up an hour later. Without a shared on-disk surface they have to re-do A's work.

ArcTeam's answer to all three is the same shape: **a swappable backend behind a `Protocol`, with the file-backed implementation doing atomic writes and `flock`-protected appends**. In tests you swap in `MemoryBackend` for speed; in production you keep `FileBackend` (or upgrade to SQLite/Postgres later — the Protocol is the same).

The four building blocks we cover here:

| Component | Persistence concern | What it stores |
|---|---|---|
| `StorageBackend` (Protocol) | Abstract interface | — |
| `FileBackend` | Disk-backed, durable | JSON records, JSONL streams |
| `MemoryBackend` | In-process, ephemeral | Dict-of-dicts |
| `TeamFileStore` | Shared workspaces between agents | Files (any format) |
| `TeamMemoryService` + `TeamMemoryConfig` | Knowledge graph, decisions, search | Markdown entities + JSONL decisions |
| `AuditLogger` | Tamper-evident chain of events | HMAC-chained JSONL |

The thread connecting them: every collection of "things that change over time" is a JSONL stream, every singleton record is an atomic JSON write, and every tamper-sensitive operation is hashed forward into the next record. That's it.

In [ ]:
# A quick sketch of the three failure modes — we'll demonstrate them concretely below.
failure_modes = [
    ("Crash mid-poll", "Cursor must survive process death", "FileBackend record"),
    ("Audit reconstruction", "Chain must be tamper-evident", "AuditLogger.verify_chain()"),
    ("Shared workspace handoff", "Files exchanged out-of-band", "TeamFileStore"),
]
for name, requirement, mechanism in failure_modes:
    print(f"  {name:30s}  →  {requirement:40s}  via  {mechanism}")

## 3. `StorageBackend` Protocol + `MemoryBackend` (in-process)

`StorageBackend` is a `runtime_checkable` `Protocol` defined in `arcteam.storage`. It declares the entire surface every persistence component depends on:

- **Records** (singleton JSON): `read`, `write`, `delete`, `exists`, `query`, `list_keys`
- **Streams** (append-only JSONL): `append`, `append_auto_seq`, `read_stream`, `read_last`, `get_stream_end_byte_pos`

Two implementations ship in the box. `MemoryBackend` keeps everything in dicts — ideal for unit tests because it's instantaneous and process-local. `FileBackend` writes JSON files for records and JSONL files for streams. They are interchangeable: every test in `tests/unit/test_storage.py` is parametrized over both.

In [ ]:
# StorageBackend is a runtime_checkable Protocol — both backends satisfy it.
mem = MemoryBackend()
fb = FileBackend(root=fresh_tmp("protocol"))

print("MemoryBackend is a StorageBackend:", isinstance(mem, StorageBackend))
print("FileBackend   is a StorageBackend:", isinstance(fb, StorageBackend))
print("Same surface — code that takes a StorageBackend works with either.")

Records first. `write` then `read` gives you back exactly what you wrote — everything goes through `json.dumps` so the data must be JSON-serializable.

In [ ]:
import asyncio


async def demo_record_basics(backend: StorageBackend) -> None:
    await backend.write(
        "entities",
        "agent_alice",
        {"id": "agent_alice", "role": "analyst"},
    )
    print(" exists:", await backend.exists("entities", "agent_alice"))
    print(" read  :", await backend.read("entities", "agent_alice"))
    print(" miss  :", await backend.read("entities", "agent_unknown"))


print("MemoryBackend:")
await demo_record_basics(mem)
print("\nFileBackend:")
await demo_record_basics(fb)

Streams are append-only. `append_auto_seq` is the safer of the two append paths — it assigns a monotonic `seq` under a lock and returns it back. That's how the messaging layer guarantees no two messages collide on the same sequence number even across processes.

In [ ]:
async def demo_stream_basics(backend: StorageBackend) -> None:
    for body in ["hello", "world", "again"]:
        seq, offset = await backend.append_auto_seq(
            "messages",
            "channel.ops",
            {"body": body},
        )
        print(f"  appended seq={seq:2d} at byte_pos={offset}")
    last = await backend.read_last("messages", "channel.ops")
    print("  last entry:", last)
    end_pos = await backend.get_stream_end_byte_pos("messages", "channel.ops")
    print("  end byte position:", end_pos)


print("MemoryBackend stream:")
await demo_stream_basics(MemoryBackend())
print("\nFileBackend stream:")
await demo_stream_basics(FileBackend(root=fresh_tmp("streams")))

Read patterns. `read_stream` accepts `after_seq` and `byte_pos` for resumption — the messaging layer's `Cursor` records both values so a reader can continue exactly where it left off without re-scanning the whole file.

In [ ]:
stream_demo = MemoryBackend()
for i in range(1, 6):
    await stream_demo.append_auto_seq("messages", "ops", {"body": f"m{i}"})

all_msgs = await stream_demo.read_stream("messages", "ops", after_seq=0)
print("All msgs:", [m["body"] for m in all_msgs])

after_two = await stream_demo.read_stream("messages", "ops", after_seq=2)
print("After seq=2:", [m["body"] for m in after_two])

first_three = await stream_demo.read_stream("messages", "ops", after_seq=0, limit=3)
print("First three with limit=3:", [m["body"] for m in first_three])

Queries do field filtering and prefix filtering on records. They're scoped per-collection; `MemoryBackend` walks its dict, `FileBackend` globs the directory.

In [ ]:
qb = MemoryBackend()
await qb.write("entities", "agent_a", {"id": "a", "type": "agent", "role": "ops"})
await qb.write("entities", "agent_b", {"id": "b", "type": "agent", "role": "dev"})
await qb.write("entities", "user_c", {"id": "c", "type": "user", "role": "ops"})

by_type = await qb.query("entities", filters={"type": "agent"})
print("agents:", [e["id"] for e in by_type])

by_role = await qb.query("entities", filters={"role": "ops"})
print("ops:   ", [e["id"] for e in by_role])

agents_only = await qb.query("entities", prefix="agent_")
print("prefix:", [e["id"] for e in agents_only])

all_keys = await qb.list_keys("entities")
print("keys:  ", all_keys)

## 4. `FileBackend` — disk persistence

`FileBackend` makes durability concrete. Three properties are non-negotiable for federal-tier deployment:

- **Atomic writes.** `write` writes to a `.tmp` file in the same directory then `os.replace()`s it onto the target. A crash mid-write leaves either the old file or the new file — never a half-written one.
- **Locked appends.** `append` and `append_auto_seq` take an `fcntl.LOCK_EX` exclusive flock before writing. Multiple processes appending to the same JSONL file get serialized; nothing interleaves.
- **Path sanitization.** Both collection names and keys go through `_sanitize`, which rejects `..`, leading `/`, and any character outside `[a-zA-Z0-9_./:-]`. Path traversal is impossible by construction.

In [ ]:
# Atomic write inspection: write a record, look at the actual file on disk.
atomic_root = fresh_tmp("atomic")
atomic = FileBackend(root=atomic_root)
await atomic.write("entities", "alice", {"id": "alice", "role": "analyst"})

# The on-disk path follows _record_path: <root>/<collection>/<key>.json
record_file = atomic_root / "entities" / "alice.json"
print("file exists       :", record_file.exists())
print("file contents     :", record_file.read_text())

# No half-written .tmp left behind.
print("any .tmp files?   :", list(record_file.parent.glob("*.tmp")))

Streams are stored as JSONL — one JSON object per line — under a directory named after the stream key. Inspecting the file shows the format directly.

In [ ]:
import json

stream_root = fresh_tmp("jsonl")
sb = FileBackend(root=stream_root)
for body in ["first", "second", "third"]:
    await sb.append_auto_seq("messages", "channel.ops", {"body": body})

stream_file = stream_root / "messages" / "channel.ops" / "00000000.log"
print("stream file:", stream_file)
for raw in stream_file.read_text().splitlines():
    print("  ", json.loads(raw))

Path sanitization. The `_sanitize` helper rejects anything that could escape the root. We let it raise so you can see exactly what's caught.

In [ ]:
sanit = FileBackend(root=fresh_tmp("sanit"))

for col, key, label in [
    ("../etc", "passwd", "dotdot in collection"),
    ("/etc", "passwd", "absolute path collection"),
    ("col", "../escape", "dotdot in key"),
    ("col with spaces", "k", "invalid characters"),
]:
    try:
        await sanit.write(col, key, {"x": 1})
        print(f"  {label:30s} -> ALLOWED (bug!)")
    except ValueError as exc:
        print(f"  {label:30s} -> rejected: {exc}")

# Nested keys with / are fine — cursor keys legitimately use them.
await sanit.write("cursors", "arc.channel.ops/agent_a1", {"seq": 5})
print("\n  legit nested key       -> OK:", await sanit.read("cursors", "arc.channel.ops/agent_a1"))

Crash recovery: a half-written line at the tail of a JSONL file should not corrupt subsequent reads. `_sync_read_stream` skips malformed JSON lines instead of raising, which means a crashed writer's torn write never blocks the next reader.

In [ ]:
torn = FileBackend(root=fresh_tmp("torn"))
await torn.append_auto_seq("streams", "s1", {"data": "good-1"})

# Manually inject a torn line — like a crashed writer would.
stream_path = torn._stream_path("streams", "s1")
with open(stream_path, "a") as f:
    f.write("{incomplete json\n")

await torn.append_auto_seq("streams", "s1", {"data": "good-2"})

results = await torn.read_stream("streams", "s1", after_seq=0)
print("recovered records (torn line skipped):")
for r in results:
    print("  ", r)

Concurrency: multiple async writers appending to the same stream get unique `seq` numbers thanks to `fcntl.LOCK_EX`. The OS serializes them; no two writers ever see the same `last_seq`.

In [ ]:
concur = FileBackend(root=fresh_tmp("concur"))


async def writer(idx: int) -> int:
    seq, _ = await concur.append_auto_seq(
        "streams",
        "shared",
        {"writer": idx},
    )
    return seq


seqs = await asyncio.gather(*(writer(i) for i in range(10)))
print("unique seq values:", sorted(seqs) == list(range(1, 11)))
print("  observed       :", sorted(seqs))

## 5. `TeamFileStore` — workspace files between agents

Sometimes agents need to exchange artifacts that aren't a fit for messages: large CSVs, generated reports, intermediate caches. `TeamFileStore` carves out a per-agent subdirectory under `<team_root>/shared/files/<agent_name>/` and copies files into it.

Two guarantees:

- **Path-within validation.** Every destination is resolved and checked against the team root. A malicious or buggy `agent_name` cannot escape the shared tree.
- **No silent overwrites.** If a destination filename already exists, a numeric suffix is appended (`report.pdf` → `report_1.pdf`).

In [ ]:
team_root = fresh_tmp("team")
store = TeamFileStore(team_root=team_root)
print("shared dir:", store.shared_dir)
print("exists yet?", store.shared_dir.exists())  # Created lazily on first store()

Stage a fake artifact and copy it in. The returned dict has the destination path, the (possibly suffixed) filename, the size, and the agent that owns it.

In [ ]:
staging = fresh_tmp("staging")
src = staging / "report.csv"
src.write_text("col_a,col_b\n1,2\n3,4\n")

info = await store.store(source_path=src, agent_name="alice")
print(info)

# Store the same name twice — second copy gets a suffix.
info2 = await store.store(source_path=src, agent_name="alice")
print(info2)

List the shared workspace. Pass an `agent_name` to filter; omit it to list every agent.

In [ ]:
# Stage another file for a different agent so the cross-agent listing has more than one row.
src_b = staging / "summary.txt"
src_b.write_text("all checks green\n")
await store.store(source_path=src_b, agent_name="bob")

alice_files = await store.list_files(agent_name="alice")
all_files = await store.list_files()

print("alice has:")
for f in alice_files:
    print("  ", f["filename"], "-", f["size"], "bytes")

print("\nteam-wide:")
for f in all_files:
    print(f"  [{f['agent']}] {f['filename']} ({f['size']} bytes)")

Path traversal rejection. An `agent_name` containing `..` or other unsafe characters is refused before any filesystem operation runs.

In [ ]:
for bad_name in ["../escape", "alice/../../etc", "with spaces", "weird;chars"]:
    try:
        await store.store(source_path=src, agent_name=bad_name)
        print(f"  {bad_name!r:30s} -> ALLOWED (bug!)")
    except ValueError:
        print(f"  {bad_name!r:30s} -> rejected")
    except FileNotFoundError:
        # If a name is borderline-valid the store call may still fail later;
        # we only care that path-escape names are caught at the validator.
        print(f"  {bad_name!r:30s} -> not blocked at validator (would fail later)")

# A missing source file is its own clean error (FileNotFoundError, not ValueError).
try:
    await store.store(source_path=staging / "does_not_exist.txt", agent_name="alice")
except FileNotFoundError as exc:
    print("\n  missing source -> FileNotFoundError:", exc)

## 6. `TeamMemoryService` + `TeamMemoryConfig`

`TeamMemoryService` is the public facade for the team knowledge graph. It coordinates an `IndexManager`, a `SearchEngine`, a `PromotionGate`, a `ClassificationChecker`, and a `MemoryStorage` — all wired up on construction from a single `TeamMemoryConfig`. We don't deep-dive each component here (that's a topic in its own right). What we cover is the **persistence-relevant surface**:

- The Null Object pattern when `enabled=False`
- How `record_decision` appends to a JSONL file under `<root>/decisions.jsonl`
- The `entities_dir` and `index_path` derived from `root`
- The `status()` snapshot used by health checks

In [ ]:
memory_root = fresh_tmp("memory")
cfg = TeamMemoryConfig(
    enabled=True,
    root=memory_root,
)
print("enabled       :", cfg.enabled)
print("root          :", cfg.root)
print("entities_dir  :", cfg.entities_dir)
print("index_path    :", cfg.index_path)
print("entity_types  :", cfg.entity_types)
print("per_entity_budget tokens:", cfg.per_entity_budget)
print("tier          :", cfg.tier)

Construct the service. `TeamMemoryService` accepts an optional `audit_logger` — when provided, every `search` call emits a `memory.search` audit event. We pass `None` here for clarity.

In [ ]:
svc = TeamMemoryService(config=cfg, audit_logger=None)

status = await svc.status()
print("status         :", status)

`record_decision` is the simplest persistence path through the memory service. It serializes a dict to JSONL under `<root>/decisions.jsonl` with a UTC timestamp and the recording agent's id. After a restart you can `tail` this file to reconstruct the decision history.

In [ ]:
await svc.record_decision(
    decision={
        "title": "Adopt FileBackend for shared storage",
        "rationale": "Survives process restart; locks ensure concurrent writers stay safe.",
        "alternatives": ["In-memory dict", "SQLite"],
    },
    agent_id="agent://alice",
)
await svc.record_decision(
    decision={"title": "Roll HMAC key quarterly", "owner": "agent://bob"},
    agent_id="agent://bob",
)

decisions_log = cfg.root / "decisions.jsonl"
print("decisions log:", decisions_log)
for line in decisions_log.read_text().splitlines():
    parsed = json.loads(line)
    print(f"  [{parsed['timestamp']}] {parsed['agent_id']}  -  {parsed.get('title', '')}")

List entities exercises the index. With a fresh root it returns an empty list — no entities have been promoted yet. `Classification.UNCLASSIFIED` is the lowest clearance tier and matches every entity that hasn't been marked as restricted.

In [ ]:
entities = await svc.list_entities(agent_classification=Classification.UNCLASSIFIED)
print("entities (empty graph):", entities)

results = await svc.search(
    query="persistence",
    agent_classification=Classification.UNCLASSIFIED,
    max_results=5,
    agent_id="agent://alice",
)
print("search results       :", results)

Null Object pattern. Disable the service via config; every method still answers — `search` returns `[]`, `promote` returns a `PromotionResult` flagged `disabled`, and `status` reports `enabled=False`. Calling code does not need to know whether memory is on.

In [ ]:
off_cfg = TeamMemoryConfig(enabled=False, root=fresh_tmp("memory-off"))
off_svc = TeamMemoryService(config=off_cfg)

print("status:", await off_svc.status())
print("search:", await off_svc.search("anything"))
print("list  :", await off_svc.list_entities())
print("get   :", await off_svc.get_entity("some_id"))

promotion = await off_svc.promote(
    entity_id="alice",
    content="# Alice\n\nAnalyst.",
    metadata=EntityMetadata(
        entity_type="person",
        entity_id="alice",
        name="Alice",
    ),
    agent_id="agent://alice",
)
print("promote:", promotion)

## 7. `AuditLogger` — chain integrity, `verify_chain`

`AuditLogger` is ArcTeam's tamper-evident log. Every record carries an `audit_seq` (monotonic counter), the operational metadata (`event_type`, `subject`, `actor_id`, `target_id`, `classification`, `detail`, `timestamp_utc`), and a `hmac_sha256` field that *chains* — each HMAC is computed over the previous record's HMAC concatenated with this record's content. Mutate any record and every subsequent HMAC stops matching.

Three operational notes:

1. **Provide a key.** Without a persistent `hmac_key` (typically loaded from `ARCTEAM_HMAC_KEY`) the logger generates a random session key and warns. The chain only verifies within the same session in that case.
2. **Always `await initialize()`.** It loads the last `audit_seq` and `prev_hmac` from disk so a restart picks up where the previous process left off.
3. **`verify_chain` is batched.** It walks the stream `_VERIFY_BATCH_SIZE` records at a time, so verifying a chain of millions of entries does not blow up memory.

In [ ]:
audit_backend = MemoryBackend()
audit = AuditLogger(audit_backend, hmac_key=b"secret-key-32-bytes-min---------")
await audit.initialize()
print("initial seq:", audit._seq)
print("prev_hmac  :", audit._prev_hmac or "(empty — first record)")

Log a few events. Each `log()` call assigns the next `audit_seq`, computes the HMAC over the prior HMAC + this record, and appends. The model fields come from `arcteam.types.AuditRecord`.

In [ ]:
events = [
    ("team.formed", "team-alpha", "created with 3 members"),
    ("message.sent", "arc.channel.ops", "task assigned"),
    ("message.delivered", "arc.channel.ops", "agent://bob received"),
    ("task.completed", "task-42", "agent://bob done"),
]
for event_type, subject, detail in events:
    await audit.log(
        event_type=event_type,
        subject=subject,
        actor_id="agent://alice",
        detail=detail,
        target_id="agent://bob",
    )

records = await audit_backend.read_stream("audit", "audit", after_seq=0)
for r in records:
    print(f"  seq={r['audit_seq']}  {r['event_type']:20s}  hmac={r['hmac_sha256'][:12]}…")

Verify the chain. With no tampering, every HMAC re-computes to the stored value and `verify_chain` returns `(True, last_seq)`.

In [ ]:
valid, last_verified = await audit.verify_chain()
print("chain valid       :", valid)
print("last verified seq :", last_verified)

Tamper test. We reach into the in-memory backend and mutate a record's `detail`. The next `verify_chain` call returns `False` along with the last seq that was still valid — investigators get an exact pointer to the breach.

In [ ]:
stored = audit_backend._streams["audit"]["audit"]
print("before tamper, record 2 detail:", stored[1]["detail"])
stored[1]["detail"] = "FORGED"

valid, last_verified = await audit.verify_chain()
print("chain valid       :", valid)
print("last good seq     :", last_verified, " (everything after this is suspect)")

Loading the HMAC key from the environment. `AuditLogger.load_hmac_key` reads `ARCTEAM_HMAC_KEY` (or any env var name you pass) and returns the bytes — `None` if not set. In production the key lives in your secret store and gets exported into the process environment at boot.

In [ ]:
# Simulate the env being set.
os.environ["ARCTEAM_HMAC_KEY"] = "persistent-key-from-secret-store"
key = AuditLogger.load_hmac_key()
print("loaded key bytes:", key)
del os.environ["ARCTEAM_HMAC_KEY"]

# Without the var set we get None — the AuditLogger constructor falls back to a
# random session key (and warns) so the chain still computes within the run.
print("unset           :", AuditLogger.load_hmac_key())

## 8. Recovery — load state from disk, replay, sanity checks

The whole point of `FileBackend` + `AuditLogger.initialize()` is that a fresh process can come up against an existing on-disk state and continue without losing or duplicating anything. This section walks the recovery flow end-to-end:

1. Process A runs, writes records and audit events.
2. Process A dies (we simulate this by dropping the in-memory references).
3. Process B starts, opens the same root, and *recovers* the prior state.

In [ ]:
# --- Process A: warm setup ---
recovery_root = fresh_tmp("recovery")
HMAC_KEY = b"persistent-test-key-32bytes-----"

backend_a = FileBackend(root=recovery_root)
audit_a = AuditLogger(backend_a, hmac_key=HMAC_KEY)
await audit_a.initialize()

# Write a few records and a few audit events.
await backend_a.write("entities", "alice", {"id": "alice", "role": "analyst"})
await backend_a.write("entities", "bob", {"id": "bob", "role": "engineer"})
for i in range(3):
    await backend_a.append_auto_seq("messages", "channel.ops", {"body": f"task-{i}"})
for i, event in enumerate(["team.formed", "member.joined", "task.dispatched"], start=1):
    await audit_a.log(
        event_type=event,
        subject=f"subject-{i}",
        actor_id="agent://alice",
        detail="warm-up",
    )

print("process A audit_seq:", audit_a._seq)
print("process A files on disk:")
for path in sorted(recovery_root.rglob("*")):
    if path.is_file():
        rel = path.relative_to(recovery_root)
        print("  ", rel, "-", path.stat().st_size, "bytes")

Simulate the process restart. Drop the references — nothing in-memory carries over.

In [ ]:
# --- Pretend process A crashed: drop references. ---
del backend_a, audit_a
print("process A objects discarded — only the on-disk state survives.")

Process B starts cold. It opens the same `recovery_root` and constructs a fresh `FileBackend` and `AuditLogger`. The crucial step is `await audit.initialize()` — that reads `read_last` from the audit stream and pulls the prior `audit_seq` and `prev_hmac` forward.

In [ ]:
# --- Process B: cold start, same root, same key ---
backend_b = FileBackend(root=recovery_root)
audit_b = AuditLogger(backend_b, hmac_key=HMAC_KEY)
await audit_b.initialize()

print("process B recovered audit_seq:", audit_b._seq)
print("process B recovered prev_hmac:", audit_b._prev_hmac[:16], "…")

# Records and streams are visible without any extra work — they live in the FS.
alice = await backend_b.read("entities", "alice")
msgs = await backend_b.read_stream("messages", "channel.ops", after_seq=0)
print("recovered alice :", alice)
print("recovered msgs  :", [m["body"] for m in msgs])

Now the new process appends its own audit events. The chain continues — the new HMAC is computed off the *prior* `prev_hmac` loaded from disk, so verification covers both pre-restart and post-restart events as one continuous chain.

In [ ]:
await audit_b.log(
    event_type="process.recovered",
    subject="process-b",
    actor_id="system",
    detail="warm restart from disk",
)
await audit_b.log(
    event_type="task.resumed",
    subject="task-1",
    actor_id="agent://alice",
    detail="continuing pre-crash work",
)

valid, last = await audit_b.verify_chain()
print("continuous chain valid?", valid)
print("final seq             :", last)

Sanity check: a different key cannot verify the same chain. This is the guarantee — the chain authenticates the writer, not just its order.

In [ ]:
wrong_key_audit = AuditLogger(backend_b, hmac_key=b"WRONG-KEY-NOT-THE-ORIGINAL-32B--")
# Note: do NOT call initialize() here — we want to verify from scratch.
valid, last = await wrong_key_audit.verify_chain()
print("wrong-key verify valid?", valid)
print("last verified seq      :", last)

Equally important: a missing entry in the middle of the chain is detected by the sequence-gap check, not just the HMAC mismatch. Verification reports `False` and tells you the last seq it saw cleanly.

In [ ]:
# Demonstrate the sequence-gap check on a fresh in-memory chain.
gap_backend = MemoryBackend()
gap_audit = AuditLogger(gap_backend, hmac_key=HMAC_KEY)
await gap_audit.initialize()
for i in range(5):
    await gap_audit.log(
        event_type=f"event_{i}",
        subject="x",
        actor_id="agent://a",
        detail=str(i),
    )
# Snip out the middle record — simulates a partial archive restore.
del gap_backend._streams["audit"]["audit"][2]
valid, last = await gap_audit.verify_chain()
print("gap-detected valid?", valid)
print("last good seq     :", last)

Memory service recovery. Re-opening the same `TeamMemoryConfig.root` from a new process picks up the prior decisions log and any persisted entities. We don't have an entities flow set up here, but the decisions log is enough to demonstrate the pattern.

In [ ]:
# Use the memory_root we set up in section 6 — same on-disk state.
svc_b = TeamMemoryService(config=TeamMemoryConfig(enabled=True, root=memory_root))
decisions_path = memory_root / "decisions.jsonl"
lines = decisions_path.read_text().splitlines() if decisions_path.exists() else []
print("decisions on disk after restart:", len(lines))
for line in lines:
    parsed = json.loads(line)
    print("  -", parsed.get("title", "<no title>"), "by", parsed["agent_id"])

# Append a post-restart decision; old decisions stay intact.
await svc_b.record_decision(
    decision={"title": "Verify chain on every boot"},
    agent_id="agent://carol",
)
post_lines = decisions_path.read_text().splitlines()
print("after appending: total lines =", len(post_lines))

## 9. Operational guidance — backups, atomic writes, perms

Persistence is only as durable as the operational practices around it. A short checklist for production deployments:

**Backups.** Snapshot the entire team root atomically — a filesystem-level copy (`rsync --link-dest`, ZFS snapshot, EBS snapshot) preserves the JSONL append order. Avoid streaming each file individually; if a snapshot crosses an in-progress append you can land mid-line. The atomic record-write pattern handles single-record writes, but not whole-tree copies.

**File permissions.** Set the team root to mode `0o700` and individual files to `0o600`. The audit log especially must be unreadable by anyone other than the service user. The `arcllm/12-security-layer.ipynb` walkthrough covers the same hardening for arcllm trace files.

**HMAC key rotation.** Rotate `ARCTEAM_HMAC_KEY` quarterly. Old chains stay verifiable with the old key, archived alongside the chain itself. New entries written under the new key. *Never* discard a key while its chain still needs to be verifiable.

**Disk space.** JSONL files grow without bound. Roll them daily (the `_stream_path` `00000000.log` filename is intentionally seq-numbered to support rotation in a later phase). Monitor inode usage as well as bytes — millions of small files exhaust inodes long before they exhaust bytes.

**Concurrent writers.** `fcntl.LOCK_EX` is per-host. If you mount the same root over NFS or run multiple containers against a shared volume, advisory locks may not be honored. The Phase 2 SQLite backend and the Phase 4 Postgres backend solve this with database-level locking — the `StorageBackend` Protocol stays the same.

In [ ]:
# Demonstrate the file-permission hardening you'd apply in production.
import stat

perm_root = fresh_tmp("perms")
perm_backend = FileBackend(root=perm_root)
await perm_backend.write("entities", "alice", {"id": "alice"})

# After writing, harden the root and the file.
os.chmod(perm_root, 0o700)
record = perm_root / "entities" / "alice.json"
os.chmod(record, 0o600)

print("root perms :", oct(stat.S_IMODE(perm_root.stat().st_mode)))
print("file perms :", oct(stat.S_IMODE(record.stat().st_mode)))

# Verify the API still works; only the OS-level access is restricted.
print("read works :", await perm_backend.read("entities", "alice"))

A second operational note: keep audit logs archived even after backend rotation. The `AuditRecord` model is a stable Pydantic schema, so a future investigator can re-parse decade-old JSONL with the same library.

In [ ]:
# Show the AuditRecord schema is round-trippable — old logs stay readable.
raw = await backend_b.read_stream("audit", "audit", after_seq=0, limit=2)
for r in raw:
    # Drop chain-specific fields; the rest is plain Pydantic-validated data.
    rec_kwargs = {k: v for k, v in r.items() if k != "seq"}
    parsed = AuditRecord(**rec_kwargs)
    print(f"  seq={parsed.audit_seq}  {parsed.event_type}  by {parsed.actor_id}")

And finally: the same recovery flow applies if you swap backends. Take a `MemoryBackend` set up for a unit test, dump its records, and import them into a `FileBackend` — the data shape is identical because both implement the same Protocol.

In [ ]:
# Cross-backend export/import — illustrates that the Protocol is the contract.
src_backend = MemoryBackend()
for i in range(3):
    await src_backend.write("entities", f"e{i}", {"id": f"e{i}", "n": i})

dst_backend = FileBackend(root=fresh_tmp("transfer"))
for key in await src_backend.list_keys("entities"):
    record = await src_backend.read("entities", key)
    if record is not None:
        await dst_backend.write("entities", key, record)

transferred = await dst_backend.list_keys("entities")
print("transferred keys:", transferred)

## 10. Summary

Persistence in ArcTeam is a federal-tier requirement implemented with a small, sharp set of components.

- **`StorageBackend` Protocol** is the contract every persistence client codes against — records (`read`/`write`/`delete`/`exists`/`query`/`list_keys`) and streams (`append`/`append_auto_seq`/`read_stream`/`read_last`/`get_stream_end_byte_pos`). Two ship-in-the-box implementations (`FileBackend`, `MemoryBackend`) interchange freely; a future SQLite/Postgres backend can drop in without touching callers.
- **`FileBackend`** delivers durability through `os.replace` atomic writes, `fcntl.LOCK_EX` JSONL appends, and `_sanitize` path validation. Torn writes from a crashed process are skipped on read; concurrent writers serialize on the lock.
- **`MemoryBackend`** mirrors the same surface in-process — used everywhere in unit tests for speed and isolation.
- **`TeamFileStore`** isolates per-agent shared workspaces under `<team_root>/shared/files/<agent>/`, validates with `_validate_path_within`, and avoids overwrites with numeric suffixes.
- **`TeamMemoryService` + `TeamMemoryConfig`** wire a knowledge-graph facade with a Null Object disabled mode, derive `entities_dir` and `index_path` from `root`, and append decisions to `<root>/decisions.jsonl` for cross-restart recall.
- **`AuditLogger`** chains HMACs across `AuditRecord` entries, calls `read_last` on `initialize()` to recover its position, and offers `verify_chain` to detect tampering, sequence gaps, and key mismatches in batches.
- **Recovery flow** is: open the same root with a fresh backend, `await audit.initialize()`, and continue. The chain spans process boundaries; old records remain Pydantic-parseable forever.
- **Operational guidance** — atomic snapshots over individual file copies, mode `0o700`/`0o600`, key rotation with archived old keys, daily JSONL rotation, awareness that `fcntl` is host-local.

Public API exercised in this notebook: `StorageBackend`, `FileBackend`, `MemoryBackend`, `TeamFileStore`, `TeamMemoryConfig`, `TeamMemoryService`, `AuditLogger`, `AuditRecord`, plus the `Classification` and `EntityMetadata` types from `arcteam.memory.types`.

**Next:** the arcteam track is now complete — `01-team-formation` (formation), `02-task-distribution` (work routing), `03-messaging-channels` (inter-agent comms), and this notebook (persistence) cover the full lifecycle. For the larger system, see `arcui/01-dashboard-bringup.ipynb` to surface team telemetry in a dashboard.